In [2]:
import os
os.chdir("/home/kjk/work") 

In [3]:
import numpy as np

a = np.array([1.0, 2.0, 3.0])  
b = np.array([4.0, 6.0, 3.0])

m = np.array([[1, 2, 3], 
              [4, 5, 6]])
print(m.shape) # 2행, 3열

print(m.sum(axis=0)) # axis=0은 열
print(m.sum(axis=1)) # axis=1은 행

(2, 3)
[5 7 9]
[ 6 15]


In [4]:
import numpy as np
from Bio.PDB import PDBParser

model = PDBParser(QUIET=True).get_structure('complex', 'data/raw/5X8M.pdb')[0]

atom1 = model['A'][56]['CA']
atom2 = model['B'][102]['CA']

print(atom1.coord)
print(atom1.coord.dtype)

print("Biopython", round(float(atom1 - atom2), 2))

diff = atom1.coord - atom2.coord
dist = np.sqrt(np.sum(diff ** 2))
print("manual", round(float(dist), 2))

print("norm", round(float(np.linalg.norm(diff)), 2))

[ 21.388 101.85   85.834]
float32
Biopython 14.62
manual 14.62
norm 14.62


In [5]:
def ca_coords(chain):
    """일반 아미노산의 CA 원자 좌표를 (N, 3) 배열로 돌려준다."""
    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    coords = np.array([r["CA"].coord for r in residues])
    return residues, coords

res_a, A = ca_coords(model["A"])
res_b, B = ca_coords(model["B"])
print(A.shape, B.shape)          # (116, 3) (214, 3)

D = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=-1)
print(D.shape)


(116, 3) (214, 3)
(116, 214)


In [17]:
flat_index = np.argmin(D)
i, j = np.unravel_index(flat_index, D.shape)
res_i = res_a[i].get_resname()
res_j = res_b[j].get_resname()

print("Chain A_B 가장 가까운 잔기")

print("Chain A:", res_a[i].get_resname(), res_a[i].id[1])

print("Chain B:", res_b[j].get_resname(), res_b[j].id[1])

print("거리", round(D[i, j], 2), "angstroms")

print("closest", "Chain A:", res_i, res_a[i].id[1],
       "Chain B:", res_j, res_b[j].id[1],
         "Distance:", round(D[i, j], 2), "angstroms")

Chain A_B 가장 가까운 잔기
Chain A: GLU 60
Chain B: TRP 102
거리 5.52 angstroms
closest Chain A: GLU 60 Chain B: TRP 102 Distance: 5.52 angstroms


In [7]:
# 과제 A: 8 Å 이내 쌍 개수
# D < 8은 True/False로 이루어진 배열이에요. 
# True는 1로 계산되니 .sum()으로 개수를 셀 수 있어요. 
# CA끼리 8 Å 이내인 쌍이 몇 개인지 출력하세요.

import numpy as np
from Bio.PDB import PDBParser

model = PDBParser(QUIET=True).get_structure('complex', 'data/raw/5X8M.pdb')[0]

def ca_coords(chain):
    """일반 아미노산의 CA 원자 좌표를 (N, 3) 배열로 돌려준다."""
    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    coords = np.array([r["CA"].coord for r in residues])
    return residues, coords

res_a, A = ca_coords(model["A"])
res_b, B = ca_coords(model["B"])
D = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=-1)

lessthan8 = D < 8

print(lessthan8.sum())


5


In [8]:
# 과제 B: 단백질 중심 좌표
# PD-L1 CA 좌표의 평균(무게중심 근사)을 구하세요. 
# 힌트: A.mean(axis=?). axis 0과 1 중 어느 쪽일까요?

A_center = np.mean(A, axis=0)

print(np.round(A_center, 2))
print(A_center.round(2))

[ 26.6  104.12  89.57]
[ 26.6  104.12  89.57]


In [9]:
# 과제 C: light chain도 분석
# chain C에 대해서도 같은 분석을 하세요. 
# 가장 가까운 쌍과 8 Å 이내 쌍 개수를 heavy chain 결과와 비교해 보세요.

def ca_coords(chain):
    """일반 아미노산의 CA 원자 좌표를 (N, 3) 배열로 돌려준다."""
    residues = [r for r in chain if r.id[0] == " " and "CA" in r]
    coords = np.array([r["CA"].coord for r in residues])
    return residues, coords

res_ab, AB = ca_coords(list(model["A"]) + list(model["B"]))
res_c, C = ca_coords(model["C"])
D1 = np.linalg.norm(C[:, None, :] - AB[None, :, :], axis=-1)

D_min_flat = np.argmin(D1)
x, y = np.unravel_index(D_min_flat, D1.shape)
res_x = res_c[x].get_resname()
res_y = res_ab[y].get_resname()

print("closest", "Chain C:", res_x, res_c[x].id[1], res_ab[y].get_parent(),
       res_y, res_ab[y].id[1])

closest Chain C: ALA 44 <Chain id=B> GLN 113


In [10]:
# 과제 D (도전): for문과 속도 비교
# 같은 D를 이중 for문으로 계산하고, time.perf_counter()로 두 방법의 실행 시간을 비교 
# 결과가 같은지는 np.allclose(D1, D2)로 확인

import time

res_a, A = ca_coords(model["A"])
res_b, B = ca_coords(model["B"])

# ── 방법 1: 브로드캐스팅 ──
start = time.perf_counter()                                 # 시작 시각 기록
D1 = np.linalg.norm(A[:, None, :] - B[None, :, :], axis=-1)
t1 = time.perf_counter() - start                            # 끝 시각 - 시작 시각 = 걸린 시간

# ── 방법 2: 이중 for문 ──
start = time.perf_counter()
D2 = np.zeros((len(A), len(B)))                  # (116, 214) 크기의 빈 표 (전부 0)
for i in range(len(A)):                          # i = 0 ~ 115 (A의 원자 번호)
    for j in range(len(B)):                      # j = 0 ~ 213 (B의 원자 번호)
        D2[i, j] = np.linalg.norm(A[i] - B[j])   # 원자 2개 거리를 표의 (i, j) 칸에 적기
t2 = time.perf_counter() - start

print(f"broadcast: {t1:.4f}초, for: {t2:.4f}초, {t2/t1:.0f}배")
print(np.allclose(D1, D2))                       # True면 두 결과가 같음

broadcast: 0.0007초, for: 0.0287초, 40배
True
